### Graph v10v11: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [1]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [2]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [3]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v17-sweep-entity-two-eval-questions']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [4]:
# Display the aggregated DataFrame
print(data_v10)

    config_evaluation.split_strategy.parameters.total_num_datapoints  \
0                                                 100                  
1                                                 100                  
2                                                 100                  
3                                                 100                  
4                                                 100                  
5                                                 100                  
6                                                 100                  
7                                                 100                  
8                                                 100                  
9                                                 100                  
10                                                100                  
11                                                100                  
12                                                100           

In [14]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std',
    'config_knowledge.entity_name',
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]


# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)


# Display the extended DataFrame
print(filtered_data)

# print a table of this
print(filtered_data.to_markdown())

   entity_name  \
0       Roomba   
1       Roomba   
2       Roomba   
3       Roomba   
4       Roomba   
5       Roomba   
6       Roomba   
7       Roomba   
8       Roomba   
9       Roomba   
10      Roomba   
11      Roomba   
12      Roomba   
13      Roomba   
14      Roomba   
15      Roomba   
16      Roomba   
17      Roomba   
18      Roomba   
19      Roomba   
20      Roomba   
21      Roomba   
22      Roomba   
23      Roomba   
24      Roomba   
25      Roomba   
26      Roomba   
27      Roomba   
28      Roomba   
29      Roomba   
30      Roomba   
31      Roomba   
32      Roomba   
33      Roomba   
34      Roomba   
35      Roomba   
36      Roomba   
37      Roomba   
38      Roomba   
39      Roomba   
40     Drizzle   
41     Drizzle   
42     Drizzle   
43     Drizzle   
44     Drizzle   
45     Drizzle   
46     Drizzle   
47     Drizzle   
48     Drizzle   
49     Drizzle   
50     Drizzle   
51     Drizzle   
52     Drizzle   
53     Drizzle   
54     Dri

/tmp/ipykernel_131880/1374537108.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_131880/1374537108.py:27: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [15]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()


# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned      0         10        100       250   500   1000
num_ordinary                                                    
0             0.002500  0.006667  0.046667  0.026667  0.09  0.29
10            0.033333  0.030000  0.030000  0.040000  0.06  0.29
2000          0.020000  0.036667  0.120000  0.230000  0.33  0.38
5000          0.013333  0.023333  0.183333  0.290000  0.46  0.26


# Heap map only for Drizzle

In [ ]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
# Filter the data for entity_name == "Drizzle"
drizzle_data = filtered_data[filtered_data['config_knowledge.entity_name'] == "Roomba"]

# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = drizzle_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[0, 10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned   0     10    100   250   500
num_ordinary                              
0             0.00  0.00  0.04  0.05  0.16
10            0.05  0.02  0.04  0.06  0.05
2000          0.04  0.03  0.05  0.07  0.49
5000          0.01  0.01  0.02  0.09  0.00


# Heat map of tinyMMLU

In [7]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of TinyMMLU Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned      0         10        100       250       500       1000
num_ordinary                                                            
0             0.631755  0.631755  0.629311  0.631755  0.631210  0.634794
10            0.631755  0.631755  0.631755  0.632768  0.631210  0.634947
2000          0.575370  0.585180  0.587632  0.598407  0.604403  0.604159
5000          0.585180  0.586848  0.579250  0.605697  0.612185  0.605672
